# Notebook 1 — Del *Playground* al Código
### Clasificación con Redes Neuronales en Python

<a href="https://colab.research.google.com/github/calderonf/NeuralNetworkSummer/blob/main/notebooks/01_Clasificacion_del_Playground_al_Codigo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

**AI Workshop: Neural Networks** — *AgeTech Latam Summer School 2026*  
Eng. Elec. Francisco Calderón — Profesor Asistente, Departamento de Electrónica, PUJ

---

## ¿Qué vas a hacer aquí?

En el [TensorFlow Playground](https://playground.tensorflow.org/) armaste una red neuronal **sin escribir código**: agregabas neuronas, capas, y veías cómo la red separaba los puntos de colores.

En este notebook vas a hacer **exactamente lo mismo, pero en Python**. La idea es que veas que:

> Cada cosa que moviste en el Playground (capas, neuronas, función de activación) es **una línea de código** que ya conoces intuitivamente.

No necesitas ser programador. Vas a ejecutar las celdas una por una con el botón ▶️ (o `Shift + Enter`) y a leer los comentarios.


## 0. Preparación del entorno

Ejecuta esta celda primero. En Google Colab casi todo ya viene instalado; esta línea solo asegura las versiones.


In [ ]:
# Instalación (en Colab suele estar todo; se ejecuta rápido)
!pip install -q scikit-learn matplotlib numpy tensorflow 2>/dev/null
print('Entorno listo')

## 1. El mismo problema del Playground: dos círculos

En el Playground, el dataset más famoso son **dos círculos anidados**: un grupo de puntos adentro y otro afuera. Una línea recta NO los puede separar, por eso necesitábamos capas ocultas.

Vamos a generar ese mismo dataset con `make_circles`.


In [ ]:
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt

# n_samples = numero de puntos ; noise = cuanto ruido (como el slider 'noise' del Playground)
X, y = make_circles(n_samples=500, noise=0.15, factor=0.5, random_state=0)

plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', alpha=0.6, edgecolors='k')
plt.title('Dataset: dos circulos (igual que en el Playground)')
plt.xlabel('Caracteristica 1 (X1)'); plt.ylabel('Caracteristica 2 (X2)')
plt.show()

print('Forma de X:', X.shape, '-> 500 puntos, cada uno con 2 numeros (X1, X2)')
print('Etiquetas y:', set(y), '-> clase 0 (azul) y clase 1 (rojo)')

## 2. Separar en entrenamiento y prueba

Igual que un estudiante: entrena con unos ejercicios (**train**) y lo evaluamos con otros que nunca vio (**test**). Así sabemos si de verdad *aprendió* o solo *memorizó*.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(f'Entrenamiento: {len(X_train)} puntos')
print(f'Prueba:        {len(X_test)} puntos')

## 3. Construir la red neuronal (la parte del Playground)

Aquí está el corazón. En el Playground elegías:

| En el Playground | En el código |
|---|---|
| Número de capas ocultas | `hidden_layer_sizes` |
| Neuronas por capa | los números dentro de `(...)` |
| Función de activación (Tanh, ReLU...) | `activation` |
| Learning rate | `learning_rate_init` |

`hidden_layer_sizes=(8, 8)` significa **2 capas ocultas de 8 neuronas cada una**. Cámbialo y experimenta.


In [ ]:
from sklearn.neural_network import MLPClassifier

red = MLPClassifier(
    hidden_layer_sizes=(8, 8),   # 2 capas ocultas, 8 neuronas c/u  <- experimenta aqui
    activation='relu',           # funcion de activacion
    learning_rate_init=0.05,     # que tan grandes son los pasos de aprendizaje
    max_iter=1000,               # cuantas veces repasa los datos (epocas)
    random_state=0
)

red.fit(X_train, y_train)   # <- aqui la red APRENDE
print('Red entrenada')

## 4. ¿Qué tan bien aprendió?

`score` = porcentaje de puntos clasificados correctamente (1.0 = 100%).


In [ ]:
print(f'Precision en entrenamiento: {red.score(X_train, y_train):.2%}')
print(f'Precision en prueba:        {red.score(X_test, y_test):.2%}')

## 5. Ver la 'frontera de decisión'

Esto es **exactamente el fondo de colores del Playground**: la región que la red considera azul y la que considera roja. Si la red aprendió bien, la frontera será un círculo.


In [ ]:
import numpy as np

def dibujar_frontera(modelo, X, y, titulo='Frontera de decision'):
    h = 0.02
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(6,6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
    plt.title(titulo); plt.xlabel('X1'); plt.ylabel('X2')
    plt.show()

dibujar_frontera(red, X, y, 'Lo que aprendio la red (comparalo con el Playground)')

## 6. Experimenta (5 minutos)

Vuelve a la celda de la **sección 3** y prueba estos cambios. Vuelve a ejecutar 3, 4 y 5 para ver el efecto:

1. `hidden_layer_sizes=()` -> **sin capas ocultas**. ¿Puede separar los círculos? (spoiler: no, igual que en el Playground).
2. `hidden_layer_sizes=(2,)` -> una sola capa con 2 neuronas.
3. `hidden_layer_sizes=(16,16,16)` -> red más grande.
4. Cambia `activation` a `'tanh'` o `'logistic'` (sigmoide).

> **Conclusión clave:** sin capas ocultas la red solo traza líneas rectas. La *no linealidad* de las capas ocultas es lo que le permite dibujar el círculo.


---
## 7. Un caso real de salud: detección de cáncer de mama

Los círculos eran de juguete. Ahora un dataset **médico real y clásico**: el *Breast Cancer Wisconsin*. A partir de mediciones de una biopsia (tamaño, textura, etc.) la red predice si un tumor es **benigno o maligno**.

Es el mismo código de antes; solo cambia el dataset. Esa es la idea central del taller: **una vez entiendes la receta, sirve para cualquier problema.**


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

datos = load_breast_cancer()
Xc, yc = datos.data, datos.target
print(f'{Xc.shape[0]} pacientes, {Xc.shape[1]} mediciones por paciente')
print('Clases:', dict(zip(datos.target_names, [int((yc==0).sum()), int((yc==1).sum())])))

# Separar
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0)

# Normalizar: poner todas las mediciones en la misma escala (MUY importante en datos reales)
escala = StandardScaler().fit(Xc_tr)
Xc_tr = escala.transform(Xc_tr)
Xc_te = escala.transform(Xc_te)
print('Datos listos y normalizados')

In [ ]:
red_med = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu',
                        max_iter=500, random_state=0)
red_med.fit(Xc_tr, yc_tr)

print(f'Precision en prueba: {red_med.score(Xc_te, yc_te):.2%}')

### Matriz de confusión: ¿en qué se equivoca?

En medicina no basta el porcentaje global: importa **qué tipo** de error comete la red (¿dice 'sano' a un enfermo?).


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred = red_med.predict(Xc_te)
cm = confusion_matrix(yc_te, pred)
ConfusionMatrixDisplay(cm, display_labels=datos.target_names).plot(cmap='Blues')
plt.title('Matriz de confusion'); plt.show()

## 8. Lo mismo con Keras / TensorFlow (opcional)

`scikit-learn` es la forma más fácil. Pero cuando los problemas crecen (imágenes, texto) se usa **Keras**. Aquí ves la MISMA red, escrita capa por capa, muy parecido a como la armaste visualmente en el Playground.


In [ ]:
import tensorflow as tf
from tensorflow import keras

modelo = keras.Sequential([
    keras.layers.Input(shape=(Xc_tr.shape[1],)),   # capa de entrada: 30 mediciones
    keras.layers.Dense(16, activation='relu'),      # capa oculta 1
    keras.layers.Dense(8,  activation='relu'),      # capa oculta 2
    keras.layers.Dense(1,  activation='sigmoid')    # salida: probabilidad 0..1
])

modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo.summary()

In [ ]:
historia = modelo.fit(Xc_tr, yc_tr, validation_split=0.1,
                      epochs=50, batch_size=16, verbose=0)

loss, acc = modelo.evaluate(Xc_te, yc_te, verbose=0)
print(f'Precision en prueba (Keras): {acc:.2%}')

# Curva de aprendizaje
plt.plot(historia.history['accuracy'], label='entrenamiento')
plt.plot(historia.history['val_accuracy'], label='validacion')
plt.xlabel('Epoca'); plt.ylabel('Precision'); plt.legend()
plt.title('Como aprendio la red epoca a epoca'); plt.show()

---
## Resumen

- Una red neuronal en Python es una **receta de pocas líneas**: define capas -> `.fit()` (aprende) -> `.score()`/`.predict()` (usa).
- Lo que movías en el Playground (capas, neuronas, activación) son **parámetros** en el código.
- Las **capas ocultas** dan la no linealidad que permite resolver problemas que una línea recta no puede.
- La MISMA receta sirve para juguetes (círculos) y para casos reales (cáncer de mama).

En el **Notebook 2** usaremos redes para **regresión**: en lugar de elegir una clase, predeciremos un **número** (progresión de una enfermedad).
